# Run 4 — RF-DETR-M Baseline

**Epic:** TTV-118 | **Config:** `rfdetr_baseline.yaml`

Primer entrenamiento de RF-DETR-Medium (Apache 2.0, candidato producción).

**Hipótesis:** DINOv2 backbone pre-entrenado rinde bien en low-data (~1K imgs)

---

## 0. Verificar GPU

In [ ]:
import torch

assert torch.cuda.is_available(), "ERROR: No GPU detectada. Ve a Runtime > Change runtime type > T4 GPU"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 1. Instalar dependencias

In [ ]:
!pip install -q roboflow "rfdetr[train,loggers]"

## 2. Montar Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_OUTPUT = '/content/drive/MyDrive/cycling-photo-ai/experiments'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)
print(f"Output dir: {DRIVE_OUTPUT}")

## 3. Descargar dataset v1 en formato COCO

In [ ]:
from roboflow import Roboflow

ROBOFLOW_API_KEY = "xOdnFACkI2vaUzBKVRic"

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("titan-ca4ce").project("titan-detection-jedpa")
version = project.version(7)

dataset = version.download("coco", location="/content/dataset_v1_coco")
print("Dataset COCO descargado")

## 4. Verificar dataset

In [ ]:
from pathlib import Path
import json

dataset_dir = Path("/content/dataset_v1_coco")

for split in ["train", "valid", "test"]:
    ann_file = dataset_dir / split / "_annotations.coco.json"
    if ann_file.exists():
        with open(ann_file) as f:
            data = json.load(f)
        print(f"{split}: {len(data['images'])} images, {len(data['annotations'])} annotations")
        if split == "train":
            cats = {c['id']: c['name'] for c in data['categories']}
            print(f"  Categories: {cats}")

## 5. Reproducibilidad

In [ ]:
from rfdetr import RFDETRMedium

RUN_NAME = "run4_rfdetr_baseline"

model = RFDETRMedium()

model.train(
    dataset_dir="/content/dataset_v1_coco",
    epochs=80,
    batch_size=4,
    grad_accum_steps=4,
    lr=5e-5,
    lr_encoder=1.5e-4,
    use_ema=True,
    early_stopping=True,
    early_stopping_patience=15,
    weight_decay=1e-4,
)

print("Entrenamiento RF-DETR completado")

## 6. Entrenar RF-DETR-M — Run 4 Baseline

Config según ADR-007:
- Resolution: 672 (múltiplo de 56)
- Batch: 4 × grad_accum 4 = efectivo 16
- EMA + Early stopping (patience 15)
- DINOv2 backbone pre-entrenado

In [ ]:
from rfdetr import RFDETRMedium

RUN_NAME = "run4_rfdetr_baseline"

model = RFDETRMedium()

model.train(
    dataset_dir="/content/dataset_v1_coco",
    epochs=80,
    batch_size=4,
    grad_accum_steps=4,
    lr=5e-5,
    lr_encoder=1.5e-4,
    resolution=672,
    use_ema=True,
    early_stopping=True,
    early_stopping_patience=15,
    weight_decay=1e-4,
)

print("Entrenamiento RF-DETR completado")

## 7. Localizar pesos y resultados

RF-DETR guarda en directorio diferente a YOLO. Exploramos para encontrar outputs.

In [ ]:
import glob

# RF-DETR puede guardar en varios lugares — buscar
print("=== Buscando archivos .pt generados ===")
for pt_file in glob.glob("/content/**/*.pt", recursive=True):
    size_mb = os.path.getsize(pt_file) / 1e6
    print(f"  {pt_file} ({size_mb:.1f} MB)")

print("\n=== Buscando logs/metrics ===")
for log_file in glob.glob("/content/**/metrics*", recursive=True):
    print(f"  {log_file}")
for log_file in glob.glob("/content/**/results*", recursive=True):
    print(f"  {log_file}")

# Checkpoints dir
print("\n=== output/ dir ===")
output_dir = Path("/content/dataset_v1_coco").parent / "output"
if output_dir.exists():
    for f in output_dir.rglob("*"):
        if f.is_file():
            print(f"  {f} ({f.stat().st_size / 1e6:.1f} MB)")
else:
    print("  output/ no encontrado, revisar logs de entrenamiento")

## 8. Evaluar con pycocotools

RF-DETR no tiene val() built-in como YOLO. Evaluamos manualmente.

In [ ]:
!pip install -q pycocotools

In [ ]:
# Inferencia sobre validation set
from rfdetr import RFDETRMedium
import json

# Cargar best checkpoint — AJUSTAR PATH según output de celda 7
# best_weights = "/content/..."  # <-- completar con path real
# eval_model = RFDETRMedium()
# eval_model.load_state_dict(torch.load(best_weights))

# Por ahora usar modelo entrenado en memoria
val_ann_path = "/content/dataset_v1_coco/valid/_annotations.coco.json"
val_img_dir = Path("/content/dataset_v1_coco/valid")

with open(val_ann_path) as f:
    val_gt = json.load(f)

predictions = []
for img_info in val_gt["images"]:
    img_path = str(val_img_dir / img_info["file_name"])
    try:
        detections = model.predict(img_path, threshold=0.25)
        # Adaptar formato según lo que devuelva predict()
        if hasattr(detections, 'xyxy'):
            # supervision format
            for j in range(len(detections.xyxy)):
                x1, y1, x2, y2 = detections.xyxy[j]
                predictions.append({
                    "image_id": img_info["id"],
                    "category_id": int(detections.class_id[j]),
                    "bbox": [float(x1), float(y1), float(x2-x1), float(y2-y1)],
                    "score": float(detections.confidence[j]),
                })
    except Exception as e:
        print(f"Error {img_info['file_name']}: {e}")

# Guardar predicciones
pred_path = "/content/rfdetr_val_predictions.json"
with open(pred_path, "w") as f:
    json.dump(predictions, f)
print(f"Predicciones: {len(predictions)} detecciones sobre {len(val_gt['images'])} imágenes")

In [ ]:
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

coco_gt = COCO(val_ann_path)
coco_dt = coco_gt.loadRes(pred_path)

coco_eval = COCOeval(coco_gt, coco_dt, "bbox")
coco_eval.evaluate()
coco_eval.accumulate()
coco_eval.summarize()

metrics = {
    "mAP@0.5:0.95": coco_eval.stats[0],
    "mAP@0.5": coco_eval.stats[1],
    "mAP@0.75": coco_eval.stats[2],
}

print("\n=== Métricas COCO ===")
for k, v in metrics.items():
    print(f"  {k}: {v:.4f}")

# Per-class
cat_ids = coco_gt.getCatIds()
cat_names = [coco_gt.loadCats(cid)[0]['name'] for cid in cat_ids]

print("\n=== Per-class AP@0.5 ===")
for cat_id, cat_name in zip(cat_ids, cat_names):
    coco_eval_cls = COCOeval(coco_gt, coco_dt, "bbox")
    coco_eval_cls.params.catIds = [cat_id]
    coco_eval_cls.evaluate()
    coco_eval_cls.accumulate()
    coco_eval_cls.summarize()
    print(f"  {cat_name:25s} AP@0.5={coco_eval_cls.stats[1]:.4f}  AP@0.5:0.95={coco_eval_cls.stats[0]:.4f}")

## 9. Guardar en Google Drive

In [ ]:
import shutil

drive_run_dir = Path(DRIVE_OUTPUT) / RUN_NAME
os.makedirs(drive_run_dir, exist_ok=True)

# Guardar predicciones y métricas
shutil.copy2(pred_path, drive_run_dir / "val_predictions.json")

# Guardar pesos — buscar .pt files
for pt_file in glob.glob("/content/**/*.pt", recursive=True):
    if "rfdetr" in pt_file.lower() or "best" in pt_file.lower() or "checkpoint" in pt_file.lower():
        shutil.copy2(pt_file, drive_run_dir / Path(pt_file).name)
        print(f"Copiado: {pt_file} → {drive_run_dir}")

# Guardar métricas como JSON
with open(drive_run_dir / "metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print(f"\nGuardado en: {drive_run_dir}")
for f in sorted(drive_run_dir.rglob("*")):
    if f.is_file():
        print(f"  {f.relative_to(drive_run_dir)} ({f.stat().st_size / 1e6:.1f} MB)")

## 10. Resumen para EXPERIMENT_LOG.md

In [ ]:
import pandas as pd

print("="*60)
print("RESUMEN PARA EXPERIMENT_LOG.md")
print("="*60)
print(f"\n### Run 4 — RF-DETR-M Baseline")
print(f"- **Fecha:** {pd.Timestamp.now().strftime('%Y-%m-%d')}")
print(f"- **Config:** rfdetr_baseline.yaml")
print(f"- **Dataset:** v1 (sin flip), Roboflow v7, formato COCO")
print(f"- **GPU:** {torch.cuda.get_device_name(0)}")
print(f"- **Arch:** RF-DETR-Medium (DINOv2 backbone, Apache 2.0)")
print(f"- **Resolution:** 672")
print(f"")
print(f"| Métrica | Valor |")
print(f"|---|---|")
for k, v in metrics.items():
    print(f"| {k} | {v:.4f} |")
print(f"")
print(f"**Per-class AP@0.5:**")
print(f"")
print(f"| Clase | AP@0.5 |")
print(f"|---|---|")
for cat_id, cat_name in zip(cat_ids, cat_names):
    coco_eval_cls = COCOeval(coco_gt, coco_dt, "bbox")
    coco_eval_cls.params.catIds = [cat_id]
    coco_eval_cls.evaluate()
    coco_eval_cls.accumulate()
    coco_eval_cls.summarize()
    print(f"| {cat_name} | {coco_eval_cls.stats[1]:.4f} |")
print(f"\nPesos guardados en: {drive_run_dir}")